# 02 · Pré-processamento

Log-retornos → split temporal → normalização (scaler só no treino) → janelas deslizantes. Lógica em `src/preprocessing.py`; decisões em `docs/adr/0001-split-temporal-antes-do-scaler.md` e `0002`/`0003`.

## Setup

In [ ]:
# --- Colab ---
# !git clone https://github.com/Cerne17/NeuraTrade.git
# %cd NeuraTrade
# !pip install -e .

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import CONFIG, set_seeds
from src import data, preprocessing as pp

set_seeds()
TICKERS = CONFIG['tickers']
series = data.load_all()
{t: df.shape for t, df in series.items()}

## 1. Log-retornos (issue #9)

$r_t = \ln(P_t / P_{t-1})$ sobre o `Close` ajustado. Invariante a escala multiplicativa, logo imune a splits/grupamentos já aplicados.

In [ ]:
rets = {t: pp.log_returns(df) for t, df in series.items()}
pd.DataFrame({t: r.describe() for t, r in rets.items()})

## 2. Split temporal ANTES de normalizar (issue #10)

Treino até `train_end` (inclusive), teste estritamente depois. Sem embaralhar.

In [ ]:
splits = {t: pp.temporal_split(r) for t, r in rets.items()}
for t, (tr, te) in splits.items():
    print(f'{t}: treino={len(tr)} ({tr.index.min().date()}..{tr.index.max().date()})'
          f'  teste={len(te)} ({te.index.min().date()}..{te.index.max().date()})')

## 3. MinMaxScaler ajustado só no treino (issue #11)

`.fit()` no treino, `.transform()` em ambos. **Valores de teste podem sair de [0,1]** — extremos pós-2020 fora do range de treino. Comportamento esperado: é onde moram as anomalias.

In [ ]:
for t, (tr, te) in splits.items():
    sc = pp.fit_scaler(tr)
    te_scaled = pp.apply_scaler(sc, te)
    fora = int(((te_scaled < 0) | (te_scaled > 1)).sum())
    print(f'{t}: treino range=[{sc.data_min_[0]:.4f},{sc.data_max_[0]:.4f}]  '
          f'teste fora de [0,1]: {fora} pontos ({100*fora/len(te_scaled):.1f}%)')

### Visualização do split e da escala

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, t in zip(axes.ravel(), TICKERS):
    tr, te = splits[t]
    ax.plot(tr.index, tr.values, lw=0.6, label='treino')
    ax.plot(te.index, te.values, lw=0.6, label='teste')
    ax.axvline(tr.index.max(), color='red', ls='--', lw=1)
    ax.set_title(t); ax.legend(loc='upper left', fontsize=8)
fig.suptitle('Log-retornos: treino vs. teste (tracejado = fim do treino)')
fig.tight_layout(); plt.show()

## 4. Janelas deslizantes (issue #12)

Tensor `(n, window_size, 1)`, geradas **dentro** de cada partição (nunca cruzam o split). `window_size=30` (ADR-0002). A validação de `window_size` por estabilidade da loss ocorre no treino (M3).

In [ ]:
pre = {t: pp.preprocess_ticker(df) for t, df in series.items()}
resumo = pd.DataFrame({
    t: {'X_train': out['X_train'].shape, 'X_test': out['X_test'].shape}
    for t, out in pre.items()
}).T
resumo

In [ ]:
# Sanidade: janela = sequencia contigua de retornos normalizados
t0 = TICKERS[0]
X = pre[t0]['X_train']
print(f'{t0}: {X.shape[0]} janelas de {X.shape[1]} passos, {X.shape[2]} feature')
print('primeira janela (10 primeiros valores):', X[0, :10, 0].round(3))

## Conclusões

- Feature pronta: log-retornos normalizados em janelas de 30 passos, ~2450 janelas de treino e ~1215 de teste por ativo.
- Split temporal e scaler-só-no-treino garantem ausência de vazamento (ADR-0001).
- Pontos de teste fora de [0,1] são esperados e úteis à detecção.
- AMER3 segue o mesmo pipeline, sem tratamento especial (ADR-0007, atualização M2): os saltos de 2024 são anomalias reais, não artefatos de grupamento.
- Próximo (M3): treinar o LSTM-Autoencoder e validar `window_size`/`latent_dim`.